In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

In [ ]:
X = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_position_matrix.csv"))
Y = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_velocity_matrix.csv"))
t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

X_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv"))
Y_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv"))

X.shape, Y.shape

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_3d(points, points_color, title="",
            azim=-60, elev=9,
            dot_size=100, alpha=0.8):

    x, y, z = points.T

    fig = plt.figure(figsize=(6, 6), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")
    fig.suptitle(title, size=14)

    # Scatter dots only
    ax.scatter(x, y, z, c=points_color,
               s=dot_size, alpha=alpha,
               edgecolors='none')

    # Clean style: no ticks, no grid, no background
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.set_facecolor("white")
    ax.grid(False)

    # Adjust viewing angle
    ax.view_init(elev=elev, azim=azim)

    plt.show()

plot_3d(X_gt,t)
plot_3d(X,t)

In [ ]:
import umap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def add_2d_scatter(ax, points, points_color, *,
                   cmap='viridis',
                   categorical_threshold=20,
                   force_discrete=False, force_continuous=False,
                   color_map=None,
                   vmin=None, vmax=None,
                   **kwargs):
    x, y = points.T
    points_color = np.asarray(points_color)

    # Decide discrete vs continuous
    if force_discrete and force_continuous:
        raise ValueError("Choose only one of force_discrete / force_continuous.")
    if force_discrete:
        is_discrete = True
    elif force_continuous:
        is_discrete = False
    else:
        is_stringish = points_color.dtype.kind in {"U", "S", "O"}
        is_integer = np.issubdtype(points_color.dtype, np.integer)
        n_unique = len(np.unique(points_color))
        is_discrete = is_stringish or (is_integer and n_unique <= categorical_threshold)

    if is_discrete:
        unique_labels = np.unique(points_color)

        if color_map is not None:
            base_cmap = plt.get_cmap('tab10', len(unique_labels))
            default_map = {lab: mcolors.to_hex(base_cmap(i)) for i, lab in enumerate(unique_labels)}
            colour_map = {**default_map, **color_map}
        else:
            base_cmap = plt.get_cmap('tab10', len(unique_labels))
            colour_map = {lab: mcolors.to_hex(base_cmap(i)) for i, lab in enumerate(unique_labels)}

        mapped_colours = np.array([colour_map[lab] for lab in points_color])
        ax.scatter(x, y, color=mapped_colours, edgecolors='none', **kwargs)

    else:
        if vmin is None: vmin = np.nanmin(points_color)
        if vmax is None: vmax = np.nanmax(points_color)
        ax.scatter(x, y, c=points_color, cmap=cmap,
                   vmin=vmin, vmax=vmax, edgecolors='none', **kwargs)

    # No legend, no colorbar — minimalist style
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_frame_on(False)


def plot_2d(points, points_color, title="", color_map=None,
            figsize=(3, 3), **kwargs):
    fig, ax = plt.subplots(figsize=figsize, facecolor="white", constrained_layout=True)
    if title:
        fig.suptitle(title, size=12)

    add_2d_scatter(ax, points, points_color,
                   color_map=color_map,
                   **kwargs)

    plt.show()


umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)

In [ ]:
plot_2d(X_2d, t, "", s=70)

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=1000)
tps.fit(X, dof=30)
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)